# BLS Monthly Data Pull
**Project:** CS689 Term Project – Traditional vs. Digital Media Consumption  
**Purpose:** Pull monthly employment/unemployment data from the Bureau of Labor Statistics public API to replace the annual-granularity web table used in Update 1.

**Series IDs (from Update 1):**
- `CES5051300001` – All employees, broadcasting & telecommunications (monthly, thousands)
- `LNU04034176`   – Unemployment rate, information sector workers (monthly, percent)

**API Docs:** https://www.bls.gov/developers/api_python.htm  
**No API key required** for up to 25 series / 500 calls per day. Registration gives 500 series / 25,000 calls.

In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime

# ── Configuration ────────────────────────────────────────────────────────────
BLS_API_URL = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

SERIES_IDS = {
    'CES5051300001': 'Broadcasting & Telecom Employment (thousands)',
    'LNU04034176':   'Information Sector Unemployment Rate (%)'
}

START_YEAR = '2010'   # captures pre-streaming era baseline
END_YEAR   = '2024'   # most recent complete year

# Optional: paste your BLS API key here for higher rate limits
# Register free at https://data.bls.gov/registrationEngine/
API_KEY = None        # e.g. 'abc123yourkeyhere'

In [ ]:
# ── API Request ───────────────────────────────────────────────────────────────
payload = {
    'seriesid':  list(SERIES_IDS.keys()),
    'startyear': START_YEAR,
    'endyear':   END_YEAR,
    'calculations': True,   # includes month-over-month and year-over-year changes
    'annualaverage': True   # includes annual average row per series
}

if API_KEY:
    payload['registrationkey'] = API_KEY

headers = {'Content-type': 'application/json'}

response = requests.post(
    BLS_API_URL,
    data=json.dumps(payload),
    headers=headers
)

data = response.json()
print(f'Status:  {data["status"]}')
print(f'Message: {data.get("message", "none")}')
print(f'Series returned: {len(data["Results"]["series"])}')

In [ ]:
# ── Parse Response into a DataFrame ──────────────────────────────────────────
def parse_bls_series(series_obj, label):
    """Flatten one BLS series into a tidy DataFrame."""
    rows = []
    for item in series_obj['data']:
        # period: 'M01'..'M12' = monthly, 'M13' = annual average
        period = item['period']
        if period == 'M13':
            month_num = None          # annual average row — keep but flag it
        else:
            month_num = int(period[1:])

        row = {
            'series_id':    series_obj['seriesID'],
            'series_label': label,
            'year':         int(item['year']),
            'month_num':    month_num,
            'period':       period,
            'value':        float(item['value']),
            'footnotes':    ', '.join(
                                [f['text'] for f in item.get('footnotes', []) if f]
                            ) or None
        }

        # Month-over-month and year-over-year changes (only present with calculations=True)
        calcs = item.get('calculations', {}).get('pct_changes', {})
        row['pct_change_1m']  = calcs.get('1')    # vs prior month
        row['pct_change_3m']  = calcs.get('3')    # vs 3 months ago
        row['pct_change_12m'] = calcs.get('12')   # vs same month prior year

        rows.append(row)

    df = pd.DataFrame(rows)

    # Build a proper date column for monthly rows
    monthly = df[df['month_num'].notna()].copy()
    monthly['date'] = pd.to_datetime(
        monthly['year'].astype(str) + '-' + monthly['month_num'].astype(int).astype(str).str.zfill(2) + '-01'
    )
    annual = df[df['month_num'].isna()].copy()
    annual['date'] = None

    return pd.concat([monthly, annual], ignore_index=True).sort_values(['year', 'month_num'])


# Parse all returned series
all_frames = []
for series in data['Results']['series']:
    sid   = series['seriesID']
    label = SERIES_IDS.get(sid, sid)
    df    = parse_bls_series(series, label)
    all_frames.append(df)
    print(f'{sid} ({label}): {len(df)} rows')

bls_raw = pd.concat(all_frames, ignore_index=True)
print(f'\nTotal rows combined: {len(bls_raw)}')

In [ ]:
# ── Preview the raw pulled data ───────────────────────────────────────────────
print('=== Raw BLS monthly data (first 15 rows) ===')
print(bls_raw.head(15).to_string())

print('\n=== Null / missing value counts ===')
print(bls_raw.isnull().sum())

print('\n=== Data types ===')
print(bls_raw.dtypes)

---
## Data Quality / Messiness
The raw API response has several issues that require transformation before loading into the warehouse — exactly what the professor noted is missing from the Kaggle sources.

In [ ]:
# ── Issue 1: footnotes contain data quality flags ─────────────────────────────
# BLS uses footnote codes like 'R' (revised), 'P' (preliminary) embedded in text
print('=== Rows with footnotes (data quality flags) ===')
flagged = bls_raw[bls_raw['footnotes'].notna()]
print(flagged[['series_id', 'year', 'period', 'value', 'footnotes']].to_string())
print(f'\n{len(flagged)} flagged rows out of {len(bls_raw)} total ({100*len(flagged)/len(bls_raw):.1f}%)')

In [ ]:
# ── Issue 2: pct_change columns are strings, not floats ───────────────────────
print('=== pct_change column types before fix ===')
print(bls_raw[['pct_change_1m', 'pct_change_3m', 'pct_change_12m']].dtypes)

for col in ['pct_change_1m', 'pct_change_3m', 'pct_change_12m']:
    bls_raw[col] = pd.to_numeric(bls_raw[col], errors='coerce')

print('\n=== pct_change column types after fix ===')
print(bls_raw[['pct_change_1m', 'pct_change_3m', 'pct_change_12m']].dtypes)

---
## Cleaned Output

In [ ]:
# ── Separate monthly from annual average rows ─────────────────────────────────
bls_monthly = bls_raw[bls_raw['period'] != 'M13'].copy()
bls_annual  = bls_raw[bls_raw['period'] == 'M13'].copy()

print(f'Monthly rows: {len(bls_monthly)}')
print(f'Annual average rows: {len(bls_annual)}')

print('\n=== Monthly sample (Broadcasting employment) ===')
emp = bls_monthly[bls_monthly['series_id'] == 'CES5051300001']
print(emp[['date', 'value', 'pct_change_1m', 'pct_change_12m', 'footnotes']].head(12).to_string())

In [ ]:
# ── Quick visual: employment trend over time ──────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

for ax, (sid, label) in zip(axes, SERIES_IDS.items()):
    subset = bls_monthly[bls_monthly['series_id'] == sid].dropna(subset=['date'])
    ax.plot(subset['date'], subset['value'], linewidth=1.5, color='steelblue')
    ax.set_title(f'{label}\n({sid})', fontsize=11)
    ax.set_ylabel('Value')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.3)

plt.suptitle('BLS Monthly Data – Traditional Media Employment\n(2010–2024)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('Data Sources copy/bls_monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to Data Sources copy/bls_monthly_trends.png')

In [ ]:
# ── Save cleaned files ────────────────────────────────────────────────────────
bls_monthly.to_csv('Data Sources copy/bls_monthly_employment.csv', index=False)
bls_annual.to_csv('Data Sources copy/bls_annual_averages.csv', index=False)

print('Saved:')
print(f'  Data Sources copy/bls_monthly_employment.csv  ({len(bls_monthly)} rows)')
print(f'  Data Sources copy/bls_annual_averages.csv     ({len(bls_annual)} rows)')
print()
print('Columns in monthly file:')
print(list(bls_monthly.columns))